# P2 | Receive the validator outputs
Operations wants to know whether the packaged run reproduced January's P1 report. Upload the three CSVs and run_record.json into your P1 Volume under p2/initial. For the supplied recovery files use p2/recovery. Reuse your actual P1 catalog and schema.

In [ ]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "bdcc_p1")
dbutils.widgets.text("run_folder", "initial")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
run_folder = dbutils.widgets.get("run_folder")
base = f"/Volumes/{catalog}/{schema}/p1_files/p2/{run_folder}"
print(base)
display(dbutils.fs.ls(base))

### Can we account for every delivery?
Read the three outputs as strings to preserve identifiers and raw rejected values. Each delivery belongs to exactly one group. Compare the loaded files with the validator's run record.

In [ ]:
frames = {}
record = spark.read.option("multiLine", True).json(f"{base}/run_record.json").first()
assert record.status == "complete", "Check the run record."
for group in ["accepted", "rejected", "duplicates"]:
    df = spark.read.option("header", True).option("inferSchema", False).csv(f"{base}/{group}.csv")
    df.createOrReplaceTempView("p2_" + group)
    frames[group] = df.count()
    assert frames[group] == record.counts[group], (group, frames[group], record.counts[group])
assert sum(frames.values()) == record.counts.delivered
print(frames, "delivered:", record.counts.delivered)
print("Input SHA-256:", record.input_sha256, "image:", record.image_id)

In [ ]:
display(spark.sql("""
SELECT reason, count(*) AS rows_rejected
FROM p2_rejected GROUP BY reason ORDER BY rows_rejected DESC, reason
"""))

### Does P1's business answer agree?
The accepted file contains several approved and declined operations and several currencies. Reapply P1's definition. Keep integer cents for the comparison.

In [ ]:
query = """
SELECT count(*) AS purchases,
       sum(cast(amount_minor AS BIGINT)) AS cents,
       sum(CASE WHEN cast(amount_minor AS BIGINT) > 100000 THEN 1 ELSE 0 END) AS large
FROM p2_accepted
WHERE operation_cd = 'PURCHASE' AND response_cd = 'APP' AND currency_cd = 'EUR'
"""
observed = spark.sql(query).first()
print(observed.asDict())
from decimal import Decimal
print("Approved EUR purchase total:", Decimal(observed.cents) / 100)

In [ ]:
# Expected values from the P1 instructor baseline, January initial file only.
assert record.input_sha256 == "454efcc2e80be19e0ed99fe2c1e20f18f86f627dc7a3a04f39ef5a1b6e3a1ce2", "Wrong input snapshot"
assert frames == {"accepted": 150000, "rejected": 150, "duplicates": 200}
assert tuple(observed) == (120487, 1038670215, 384), "Investigate the population and amount units"
print("The January outputs reconcile with P1.")

### Exit ticket
Which run could produce rejected rows and still succeed? Which piece of evidence tells you that a rerun used the same input? Keep the run record and one line explaining what you would hold constant before timing this program in P3.